In [2]:
import numpy as np
import os
import fnmatch

In [8]:
def filter_files(path, pattern):
    files_mu_sigma = []
    # Iterate through the files in the specified directory
    for filename in os.listdir(path):
        if fnmatch.fnmatch(filename, pattern):
            parts = filename.split('_')
            files_mu_sigma.append((filename, int(parts[10]), float(parts[14]))) 
    sorted_pairs = sorted(files_mu_sigma, key=lambda x: (x[1], x[2]))  # Sort by mu value, sigma value, and S value
    return sorted_pairs

In [4]:
def group_files_by_params(files):
    grouped_files = {}
    for file_info in files:
        filename, S, sigma = file_info
        key = (S, sigma)
        if key not in grouped_files:
            grouped_files[key] = []
        grouped_files[key].append(filename)
    return grouped_files

In [5]:
def extract_data_from_file(path, filename):
    fin = open(f'{path}/{filename}', 'r')
    fin.readline()  # Skip the header line
    line = fin.readline().split()
    if line[0] == 'Divergence':
        divergence = True
        convergence = False
        final_delta = np.inf
    elif line[0] == 'Convergence':
        divergence = False
        convergence = True
        final_delta = float(line[7])
    elif line[0] == 'Maximum':
        divergence = False
        convergence = False
        final_delta = float(line[7])
    else:
        raise ValueError(f"Unexpected line format in file {filename}: {line}")
    fin.close()
    return convergence, divergence, final_delta

In [13]:
def average_final_delta(pathout, fileout, path, pattern, ndigits=6):
    files = filter_files(path, pattern)
    grouped_files = group_files_by_params(files)
    fout = open(f'{pathout}/{fileout}', 'w')
    fout.write('#sigma(X10^3) S delta error_delta nsamples fraction_diverged fraction_converged fraction_fluctuates\n')
    for S, sigma in grouped_files.keys():
        filenames = grouped_files[(S, sigma)]
        av_delta = 0
        av_delta_sqr = 0
        nsamples = len(filenames)
        n_diverged = 0
        n_converged = 0
        n_fluctuates = 0
        counter = 0
        for filename in filenames:
            convergence, divergence, final_delta = extract_data_from_file(path, filename)
            if divergence:
                n_diverged += 1
            elif convergence:
                n_converged += 1
                av_delta += final_delta
                av_delta_sqr += final_delta**2
                counter += 1
            elif not convergence and not divergence:
                n_fluctuates += 1
                av_delta += final_delta
                av_delta_sqr += final_delta**2
                counter += 1
        if counter > 0:
            av_delta /= counter
            av_delta_sqr /= counter
            error_delta = np.sqrt(av_delta_sqr - av_delta**2)
        else:
            av_delta = np.nan
            error_delta = np.nan
        fraction_diverged = n_diverged / nsamples
        fraction_converged = n_converged / nsamples
        fraction_fluctuates = n_fluctuates / nsamples
        fout.write(f'{int(sigma*1000)} {S} {av_delta} {error_delta} {nsamples} {fraction_diverged:.{ndigits}f} {fraction_converged:.{ndigits}f} {fraction_fluctuates:.{ndigits}f}\n')
        print(f'Processed file: sigma={sigma}, S={S}')
    fout.close()

In [11]:
# path = '/mnt/d/Research/Ecology/Results/IBMF/PopDyn/AllData'
# pathout = '/mnt/d/Research/Ecology/Results/IBMF/PopDyn'
path = '/media/david/Seagate Expansion Drive/Salva/Salva_Data_Investigacion/Grupo_de_investigacion/Ecology/Results/GECaM/AllData'
pathout = '/media/david/Seagate Expansion Drive/Salva/Salva_Data_Investigacion/Grupo_de_investigacion/Ecology/Results/GECaM'

T = "0.000"
mu = "0.000"
tol = "1e-6"

In [14]:
fileout = f'GECaM_popdyn_RRG_T_{T}_PD_Lotka_Volterra_mu_{mu}_tol_{tol}.txt'
pattern = f'Out_GECaM_popdyn_RRG_T_{T}_PD_Lotka_Volterra_S_*_mu_{mu}_sigma_*_tol_{tol}_seed_*.txt'
average_final_delta(pathout, fileout, path, pattern)

Processed file: sigma=0.15, S=8192
Processed file: sigma=0.25, S=8192
Processed file: sigma=0.35, S=8192
Processed file: sigma=0.45, S=8192
Processed file: sigma=0.15, S=32768
Processed file: sigma=0.25, S=32768
Processed file: sigma=0.35, S=32768
Processed file: sigma=0.45, S=32768
Processed file: sigma=0.15, S=131072
Processed file: sigma=0.25, S=131072
Processed file: sigma=0.35, S=131072
Processed file: sigma=0.45, S=131072
Processed file: sigma=0.15, S=524288
Processed file: sigma=0.25, S=524288
Processed file: sigma=0.35, S=524288
Processed file: sigma=0.45, S=524288
Processed file: sigma=0.15, S=2097152
Processed file: sigma=0.25, S=2097152
Processed file: sigma=0.35, S=2097152
Processed file: sigma=0.45, S=2097152
Processed file: sigma=0.15, S=8388608
Processed file: sigma=0.25, S=8388608
Processed file: sigma=0.35, S=8388608
Processed file: sigma=0.45, S=8388608
